In [1]:
%pwd

'/kaggle/working'

In [2]:
!pip install gdown

In [3]:
! gdown  1hiBM9zdUHNj98xmKgDabKx8QbDpECrB7

Downloading...
From (original): https://drive.google.com/uc?id=1hiBM9zdUHNj98xmKgDabKx8QbDpECrB7
From (redirected): https://drive.google.com/uc?id=1hiBM9zdUHNj98xmKgDabKx8QbDpECrB7&confirm=t&uuid=f46a2ec0-acd1-4e51-9c63-4571aedfdb0e
To: /kaggle/working/results.zip
100%|██████████████████████████████████████| 3.90G/3.90G [00:58<00:00, 66.9MB/s]


In [4]:
! unzip -o results.zip 
! rm -r results.zip

Archive:  results.zip
  inflating: .virtual_documents/__notebook_source__.ipynb  
  inflating: Fine-tune/README.md     
  inflating: Fine-tune/configs/detection_config.yml  
  inflating: Fine-tune/configs/recognition_config.yml  
  inflating: Fine-tune/documentation/error_handling.md  
  inflating: Fine-tune/documentation/get_your_OCR_model_finally_customized.gif  
  inflating: Fine-tune/inference_results/det_res_11.jpeg  
  inflating: Fine-tune/inference_results/det_results.txt  
  inflating: Fine-tune/log_output/benchmark_detection.log  
  inflating: Fine-tune/main.ipynb    
  inflating: Fine-tune/output/recognition/config.yml  
  inflating: Fine-tune/output/recognition/train.log  
  inflating: Fine-tune/output/vn_detection/best_accuracy.pdopt  
  inflating: Fine-tune/output/vn_detection/best_accuracy.pdparams  
  inflating: Fine-tune/output/vn_detection/best_accuracy.states  
  inflating: Fine-tune/output/vn_detection/best_model/model.pdopt  
  inflating: Fine-tune/output/vn_detecti

In [5]:
%cd /kaggle/working/Fine-tune

/kaggle/working/Fine-tune


In [6]:
%%writefile ./configs/recognition_config.yml
Global:
  debug: false
  use_gpu: true
  epoch_num: 100
  log_smooth_window: 20
  print_batch_step: 10
  save_model_dir: /kaggle/working/Fine-tune/output/vn_recognition_2
  save_epoch_step: 100
  eval_batch_step: [0, 2000]
  cal_metric_during_train: true
  pretrained_model: pretrained_models/recognition/en_PP-OCRv3_rec_train/best_accuracy.pdparams
  checkpoints: 
  save_inference_dir:
  use_visualdl: false
  infer_img: 
  character_dict_path: /kaggle/working/Fine-tune/vn_dictionary.txt
  max_text_length: &max_text_length 25
  infer_mode: false
  use_space_char: true
  distributed: true
  save_res_path: output/vn_recognition/predicts/text.txt


Optimizer:
  name: Adam
  beta1: 0.9
  beta2: 0.999
  lr:
    name: Cosine
    learning_rate: 0.001
    warmup_epoch: 5
  regularizer:
    name: L2
    factor: 3.0e-05


Architecture:
  model_type: rec
  algorithm: SVTR
  Transform:
  Backbone:
    name: MobileNetV1Enhance
    scale: 0.5
    last_conv_stride: [1, 2]
    last_pool_type: avg
    last_pool_kernel_size: [2, 2]
  Head:
    name: MultiHead
    head_list:
      - CTCHead:
          Neck:
            name: svtr
            dims: 64
            depth: 2
            hidden_dims: 120
            use_guide: True
          Head:
            fc_decay: 0.00001
      - SARHead:
          enc_dim: 512
          max_text_length: *max_text_length

Loss:
  name: MultiLoss
  loss_config_list:
    - CTCLoss:
    - SARLoss:

PostProcess:  
  name: CTCLabelDecode

Metric:
  name: RecMetric
  main_indicator: acc
  ignore_space: False

Train:
  dataset:
    name: SimpleDataSet
    data_dir: ./
    ext_op_transform_idx: 1
    label_file_list:
    - recognition_train.txt
    transforms:
    - DecodeImage:
        img_mode: BGR
        channel_first: false
    - RecConAug:
        prob: 0.5
        ext_data_num: 2
        image_shape: [48, 320, 3]
        max_text_length: *max_text_length
    - RecAug:
    - MultiLabelEncode:
    - RecResizeImg:
        image_shape: [3, 48, 320]
    - KeepKeys:
        keep_keys:
        - image
        - label_ctc
        - label_sar
        - length
        - valid_ratio
  loader:
    shuffle: true
    batch_size_per_card: 64
    drop_last: true
    num_workers: 4
Eval:
  dataset:
    name: SimpleDataSet
    data_dir: ./
    label_file_list:
    - recognition_test.txt
    transforms:
    - DecodeImage:
        img_mode: BGR
        channel_first: false
    - MultiLabelEncode:
    - RecResizeImg:
        image_shape: [3, 48, 320]
    - KeepKeys:
        keep_keys:
        - image
        - label_ctc
        - label_sar
        - length
        - valid_ratio
  loader:
    shuffle: false
    drop_last: false
    batch_size_per_card: 32
    num_workers: 4

Overwriting ./configs/recognition_config.yml


In [7]:
%%writefile /kaggle/working/Fine-tune/configs/detection_config.yml
Global:
  infer_mode: true
  export_with_pir: true
  use_gpu: true
  use_xpu: false
  use_mlu: false
  epoch_num: 100  # CHANGED: Increase from 5 to 100 for better training
  log_smooth_window: 20
  print_batch_step: 10
  save_model_dir: output/vn_detection  # CHANGED: Better naming
  save_epoch_step: 10  # CHANGED: Save every 10 epochs instead of 1000
  eval_batch_step: [1000, 1000]  # CHANGED: Start evaluation at iter 1000, then every 1000 iters
  cal_metric_during_train: False  # CHANGED: Disable metric calculation to avoid shape errors
  pretrained_model: pretrained_models/detection/MobileNetV3_large_x0_5_pretrained.pdparams
  checkpoints: 
  save_inference_dir:
  use_visualdl: False
  infer_img: 
  save_res_path: 

Architecture:
  model_type: det
  algorithm: DB
  Transform:
  Backbone:
    name: MobileNetV3
    scale: 0.5
    model_name: large
  Neck:
    name: DBFPN
    out_channels: 256
  Head:
    name: DBHead
    k: 50

Loss:
  name: DBLoss
  balance_loss: true
  main_loss_type: DiceLoss
  alpha: 5
  beta: 10
  ohem_ratio: 3

Optimizer:
  name: Adam
  beta1: 0.9
  beta2: 0.999
  lr:
    name: Cosine
    learning_rate: 0.001
    warmup_epoch: 2
  regularizer:
    name: 'L2'
    factor: 0

PostProcess:
  name: DBPostProcess
  thresh: 0.3
  box_thresh: 0.6
  max_candidates: 1000
  unclip_ratio: 1.5

Metric:
  name: DetMetric
  main_indicator: hmean

Train:
  dataset:
    name: SimpleDataSet
    data_dir: ./
    label_file_list:
      - train_detection.txt  # CHANGED: Use your prepared file
    ratio_list: [1.0]
    transforms:
      - DecodeImage:
          img_mode: BGR
          channel_first: False
      - DetLabelEncode:
      - IaaAugment:
          augmenter_args:
            - { 'type': Fliplr, 'args': { 'p': 0.5 } }
            - { 'type': Affine, 'args': { 'rotate': [-10, 10] } }
            - { 'type': Resize, 'args': { 'size': [0.5, 3] } }
      - EastRandomCropData:
          size: [640, 640]
          max_tries: 50
          keep_ratio: true
      - MakeBorderMap:
          shrink_ratio: 0.4
          thresh_min: 0.3
          thresh_max: 0.7
      - MakeShrinkMap:
          shrink_ratio: 0.4
          min_text_size: 8
      - NormalizeImage:
          scale: 1./255.
          mean: [0.485, 0.456, 0.406]
          std: [0.229, 0.224, 0.225]
          order: 'hwc'
      - ToCHWImage:
      - KeepKeys:
          keep_keys: ['image', 'threshold_map', 'threshold_mask', 'shrink_map', 'shrink_mask']
  loader:
    shuffle: True
    drop_last: False
    batch_size_per_card: 8  # ADJUST: Reduce to 4 if GPU memory issues
    num_workers: 4
    use_shared_memory: True

Eval:
  dataset:
    name: SimpleDataSet
    data_dir: ./
    label_file_list:
      - test_detection.txt  # CHANGED: Use your prepared file
    transforms:
      - DecodeImage:
          img_mode: BGR
          channel_first: False
      - DetLabelEncode:
      - DetResizeForTest:
      - NormalizeImage:
          scale: 1./255.
          mean: [0.485, 0.456, 0.406]
          std: [0.229, 0.224, 0.225]
          order: 'hwc'
      - ToCHWImage:
      - KeepKeys:
          keep_keys: ['image', 'shape', 'polys', 'ignore_tags']
  loader:
    shuffle: False
    drop_last: False
    batch_size_per_card: 1
    num_workers: 4
    use_shared_memory: True

Overwriting /kaggle/working/Fine-tune/configs/detection_config.yml


In [8]:
!git clone https://github.com/PaddlePaddle/PaddleOCR.git

Cloning into 'PaddleOCR'...
remote: Enumerating objects: 308697, done.
remote: Counting objects: 100% (662/662), done.
remote: Compressing objects: 100% (148/148), done.
remote: Total 308697 (delta 592), reused 514 (delta 514), pack-reused 308035 (from 2)
Receiving objects: 100% (308697/308697), 1.63 GiB | 37.96 MiB/s, done.
Resolving deltas: 100% (244370/244370), done.


In [9]:
!python -m pip install paddlepaddle-gpu==3.2.2 -i https://www.paddlepaddle.org.cn/packages/stable/cu129/
!pip install -q paddleocr
!pip install -q imutils
!pip install -q pyclipper lmdb rapidfuzz paddleocr

Looking in indexes: https://www.paddlepaddle.org.cn/packages/stable/cu129/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 GB 970.2 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.6/89.6 MB 9.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 15.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 15.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 781.4/781.4 MB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 580.7/580.7 MB 1.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.9/200.9 MB 6.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.3/68.3 MB 11.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.8/331.8 MB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.4/366.4 MB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [10]:
!pwd
import os
import cv2
import json

import numpy as np
import matplotlib.pyplot as plt

from tqdm import tqdm
from PIL import Image
from imutils import perspective

# from paddleocr import PaddleOCR

/kaggle/working/Fine-tune


In [11]:
import paddle

!pip show paddleocr
print("GPU available:", paddle.device.is_compiled_with_cuda())

/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:718: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)


Name: paddleocr
Version: 3.3.2
Summary: Awesome multilingual OCR and document parsing toolkits based on PaddlePaddle
Home-page: https://github.com/PaddlePaddle/PaddleOCR
Author: 
Author-email: PaddlePaddle <paddleocr@baidu.com>
License: Apache License 2.0
Location: /usr/local/lib/python3.12/dist-packages
Requires: paddlex, PyYAML, requests, typing-extensions
Required-by: 
GPU available: True


In [12]:
paddle.utils.run_check()

Running verify PaddlePaddle program ... 


/usr/local/lib/python3.12/dist-packages/paddle/pir/math_op_patch.py:219: UserWarning: Value do not have 'place' interface for pir graph mode, try not to use it. None will be returned.
  warnings.warn(
I0112 16:01:40.410676    24 pir_interpreter.cc:1524] New Executor is Running ...
W0112 16:01:40.411157    24 gpu_resources.cc:114] Please NOTE: device: 0, GPU Compute Capability: 7.5, Driver API Version: 12.8, Runtime API Version: 12.9
I0112 16:01:40.412065    24 pir_interpreter.cc:1547] pir interpreter is running by multi-thread mode ...


PaddlePaddle works well on 1 GPU.


/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:718: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:718: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
======================= Modified FLAGS detected =======================
FLAGS(name='FLAGS_cuda_cccl_dir', current_value='/usr/local/lib/python3.12/dist-packages/paddle/../nvidia/cuda_cccl/include/', default_value='')
FLAGS(name='FLAGS_nccl_dir', current_value='/usr/local/lib/python3.12/dist-packages/paddle/../nvidia/nccl/lib', default_value='')
FLAGS(n

PaddlePaddle works well on 2 GPUs.
PaddlePaddle is installed successfully! Let's start deep learning with PaddlePaddle now.


In [13]:

# def prepare_vietnamese_detection_file(data_type, input_image_dir, label_dir, output_file):
#     """
#     Prepare PaddleOCR format annotation file for Vietnamese dataset

#     Args:
#         data_type: 'train' or 'test'
#         input_image_dir: path to train_images or test_image folder
#         label_dir: path to labels folder
#         output_file: output .txt file path
#     """
#     count = 0
#     skipped = 0

#     # Clear output file if it exists
#     if os.path.exists(output_file):
#         os.remove(output_file)

#     # Get all label files to create a mapping
#     label_files = [f for f in os.listdir(label_dir) if f.startswith('gt_') and f.endswith('.txt')]

#     # Create mapping: label_number -> label_file
#     # gt_548.txt -> 548
#     label_numbers = {}
#     for lf in label_files:
#         label_num = lf.replace('gt_', '').replace('.txt', '')
#         label_numbers[label_num] = lf

#     print(f"Found {len(label_numbers)} label files")
#     if label_numbers:
#         label_nums_int = sorted([int(k) for k in label_numbers.keys()])
#         print(f"Label numbers range: {label_nums_int[0]} to {label_nums_int[-1]}")

#     # Get all image files
#     image_files = sorted([f for f in os.listdir(input_image_dir) if f.endswith(('.jpg', '.png', '.jpeg'))])

#     print(f"Found {len(image_files)} image files")
#     print(f"Image range: {image_files[0]} to {image_files[-1]}")

#     # Show first few mappings as example
#     print("\nExample mappings (first 5):")
#     for i, img_file in enumerate(image_files[:5]):
#         img_number = img_file.replace('im', '').split('.')[0]
#         img_num_int = str(int(img_number))
#         label_file = label_numbers.get(img_num_int, "NOT FOUND")
#         print(f"  {img_file} -> {label_file}")

#     for img_file in tqdm(image_files, desc=f"Processing {data_type}"):
#         img_path = os.path.join(input_image_dir, img_file)

#         # Extract image number: im0001.jpg -> 1, im0202.jpg -> 202
#         img_number = img_file.replace('im', '').split('.')[0]
#         img_num_int = str(int(img_number))  # Remove leading zeros

#         # Try to find matching label file
#         label_file = None
#         if img_num_int in label_numbers:
#             label_file = label_numbers[img_num_int]

#         if label_file is None:
#             skipped += 1
#             continue

#         label_path = os.path.join(label_dir, label_file)

#         # Read label file
#         try:
#             with open(label_path, 'r', encoding='utf-8') as f:
#                 lines = f.readlines()
#         except Exception as e:
#             print(f"Error reading {label_file}: {e}")
#             skipped += 1
#             continue

#         annotations = []

#         for line in lines:
#             line = line.strip()
#             if not line:
#                 continue

#             try:
#                 # Parse format: x1,y1,x2,y2,x3,y3,x4,y4,text
#                 parts = line.split(',')

#                 if len(parts) < 9:  # At least 8 coordinates + text
#                     continue

#                 # Extract coordinates (first 8 values)
#                 x1, y1, x2, y2, x3, y3, x4, y4 = map(int, parts[:8])

#                 # Extract text (everything after 8th comma)
#                 text = ','.join(parts[8:])  # In case text contains commas

#                 # Skip if text is ### (illegible)
#                 # PaddleOCR uses ### to mark text regions that should be ignored during training
#                 if text.strip() == '###':
#                     text = '###'

#                 # Create annotation in PaddleOCR format
#                 annotation = {
#                     "transcription": text,
#                     "points": [[x1, y1], [x2, y2], [x3, y3], [x4, y4]]
#                 }

#                 annotations.append(annotation)

#             except Exception as e:
#                 print(f"Error parsing line in {label_file}: {line}")
#                 print(f"Error: {e}")
#                 continue

#         # Skip images with no valid annotations
#         if len(annotations) == 0:
#             skipped += 1
#             continue

#         # Write to output file in PaddleOCR format
#         # Format: image_path\t[{annotations}]\n
#         with open(output_file, 'a', encoding='utf-8') as f:
#             f.write(img_path + '\t' + json.dumps(annotations, ensure_ascii=False) + '\n')

#         count += 1

#     print(f"\n{'='*50}")
#     print(f"Dataset type: {data_type}")
#     print(f"Total images processed: {count}")
#     print(f"Skipped images: {skipped}")
#     print(f"Output file: {output_file}")
#     print(f"{'='*50}\n")

#     return count


# # Example usage
# if __name__ == "__main__":
#     # Set your base directory
#     base_dir = "/content/drive/MyDrive/Fine-Tuning-OCR-Model-with-PaddleOCR/dataset"  # Change this to your actual path

#     # Prepare training data
#     prepare_vietnamese_detection_file(
#         data_type='train',
#         input_image_dir=os.path.join(base_dir, 'train_images'),
#         label_dir=os.path.join(base_dir, 'labels'),
#         output_file='train_detection.txt'
#     )

#     # Prepare test data
#     prepare_vietnamese_detection_file(
#         data_type='test',
#         input_image_dir=os.path.join(base_dir, 'test_image'),
#         label_dir=os.path.join(base_dir, 'labels'),
#         output_file='test_detection.txt'
#     )

#     print("Data preparation completed!")
#     print("\nNext steps:")
#     print("1. Verify the output files: train_detection.txt and test_detection.txt")
#     print("2. Update your PaddleOCR config file to point to these files")
#     print("3. Start training!")

In [14]:
# !rm recognition_test.txt
# !rm recognition_train.txt

In [15]:

def four_point_transform(image, pts):
    """
    Perform perspective transform to get a rectangular crop of the text region
    """
    # Obtain a consistent order of the points
    rect = np.zeros((4, 2), dtype="float32")

    # Top-left point has smallest sum, bottom-right has largest sum
    s = pts.sum(axis=1)
    rect[0] = pts[np.argmin(s)]
    rect[2] = pts[np.argmax(s)]

    # Top-right has smallest difference, bottom-left has largest difference
    diff = np.diff(pts, axis=1)
    rect[1] = pts[np.argmin(diff)]
    rect[3] = pts[np.argmax(diff)]

    # Compute width and height of the new image
    (tl, tr, br, bl) = rect
    widthA = np.sqrt(((br[0] - bl[0]) ** 2) + ((br[1] - bl[1]) ** 2))
    widthB = np.sqrt(((tr[0] - tl[0]) ** 2) + ((tr[1] - tl[1]) ** 2))
    maxWidth = max(int(widthA), int(widthB))

    heightA = np.sqrt(((tr[0] - br[0]) ** 2) + ((tr[1] - br[1]) ** 2))
    heightB = np.sqrt(((tl[0] - bl[0]) ** 2) + ((tl[1] - bl[1]) ** 2))
    maxHeight = max(int(heightA), int(heightB))

    # Construct destination points
    dst = np.array([
        [0, 0],
        [maxWidth - 1, 0],
        [maxWidth - 1, maxHeight - 1],
        [0, maxHeight - 1]], dtype="float32")

    # Calculate perspective transform matrix and warp
    M = cv2.getPerspectiveTransform(rect, dst)
    warped = cv2.warpPerspective(image, M, (maxWidth, maxHeight))

    return warped


def prepare_vietnamese_recognition_file(data_type, input_image_dir, label_dir,
                                        output_image_dir, output_file):
    """
    Prepare recognition dataset by cropping text regions from full images

    Args:
        data_type: 'train' or 'test'
        input_image_dir: path to train_images or test_image folder
        label_dir: path to labels folder
        output_image_dir: directory to save cropped text images
        output_file: output label file path (image_path\ttext format)
    """
    # Create output directory
    if not os.path.exists(output_image_dir):
        os.makedirs(output_image_dir)

    # Clear output file if exists
    if os.path.exists(output_file):
        os.remove(output_file)

    # Get all label files
    label_files = [f for f in os.listdir(label_dir) if f.startswith('gt_') and f.endswith('.txt')]
    label_numbers = {}
    for lf in label_files:
        label_num = lf.replace('gt_', '').replace('.txt', '')
        label_numbers[label_num] = lf

    # Get all image files
    image_files = sorted([f for f in os.listdir(input_image_dir)
                         if f.endswith(('.jpg', '.png', '.jpeg'))])

    total_crops = 0
    processed_images = 0
    skipped_images = 0

    print(f"Processing {len(image_files)} images for {data_type} recognition dataset...")

    for img_file in tqdm(image_files, desc=f"Processing {data_type}"):
        img_path = os.path.join(input_image_dir, img_file)

        # Find corresponding label file
        img_number = img_file.replace('im', '').split('.')[0]
        img_num_int = str(int(img_number))

        if img_num_int not in label_numbers:
            skipped_images += 1
            continue

        label_file = label_numbers[img_num_int]
        label_path = os.path.join(label_dir, label_file)

        # Read image
        try:
            image = cv2.imread(img_path)
            if image is None:
                print(f"Failed to read image: {img_file}")
                skipped_images += 1
                continue
        except Exception as e:
            print(f"Error reading {img_file}: {e}")
            skipped_images += 1
            continue

        # Read labels
        try:
            with open(label_path, 'r', encoding='utf-8') as f:
                lines = f.readlines()
        except Exception as e:
            print(f"Error reading label {label_file}: {e}")
            skipped_images += 1
            continue

        # Process each text region in the image
        for line_idx, line in enumerate(lines):
            line = line.strip()
            if not line:
                continue

            try:
                # Parse: x1,y1,x2,y2,x3,y3,x4,y4,text
                parts = line.split(',')
                if len(parts) < 9:
                    continue

                # Extract coordinates
                x1, y1, x2, y2, x3, y3, x4, y4 = map(int, parts[:8])
                text = ','.join(parts[8:])

                # Skip illegible text marked as ###
                if text.strip() == '###':
                    continue

                # Create polygon points
                pts = np.array([[x1, y1], [x2, y2], [x3, y3], [x4, y4]], dtype="float32")

                # Perform perspective transform to get rectangular crop
                warped = four_point_transform(image, pts)

                # Skip if crop is too small
                if warped.shape[0] < 5 or warped.shape[1] < 5:
                    continue

                # Save cropped image
                crop_filename = f"crop_{total_crops:06d}.jpg"
                crop_path = os.path.join(output_image_dir, crop_filename)
                cv2.imwrite(crop_path, warped)

                # Write to label file
                with open(output_file, 'a', encoding='utf-8') as f:
                    f.write(f"{crop_path}\t{text}\n")

                total_crops += 1

            except Exception as e:
                print(f"Error processing line in {label_file}: {line}")
                print(f"Error: {e}")
                continue

        processed_images += 1

    print(f"\n{'='*50}")
    print(f"Dataset type: {data_type}")
    print(f"Total images processed: {processed_images}")
    print(f"Skipped images: {skipped_images}")
    print(f"Total text crops created: {total_crops}")
    print(f"Output directory: {output_image_dir}")
    print(f"Output file: {output_file}")
    print(f"{'='*50}\n")

    return total_crops


# Example usage
if __name__ == "__main__":
    base_dir = "/kaggle/input/dataset/dataset"  # Change this

    # Prepare training recognition data
    train_crops = prepare_vietnamese_recognition_file(
        data_type='train',
        input_image_dir=os.path.join(base_dir, 'train_images'),
        label_dir=os.path.join(base_dir, 'labels'),
        output_image_dir='crnn_train',
        output_file='recognition_train.txt'
    )

    # Prepare test recognition data
    test_crops = prepare_vietnamese_recognition_file(
        data_type='test',
        input_image_dir=os.path.join(base_dir, 'test_image'),
        label_dir=os.path.join(base_dir, 'labels'),
        output_image_dir='crnn_test',
        output_file='recognition_test.txt'
    )

    print("Recognition data preparation completed!")
    print(f"Training crops: {train_crops}")
    print(f"Test crops: {test_crops}")
    print("\nYou now have:")
    print("1. Detection data: train_detection.txt, test_detection.txt")
    print("2. Recognition data: recognition_train.txt, recognition_test.txt")
    print("3. Cropped images: crnn_train/, crnn_test/")

Processing 1200 images for train recognition dataset...


Processing train: 100%|██████████| 1200/1200 [00:59<00:00, 20.13it/s]



Dataset type: train
Total images processed: 1200
Skipped images: 0
Total text crops created: 25693
Output directory: crnn_train
Output file: recognition_train.txt

Processing 300 images for test recognition dataset...


Processing test: 100%|██████████| 300/300 [00:14<00:00, 20.83it/s]


Dataset type: test
Total images processed: 300
Skipped images: 0
Total text crops created: 7206
Output directory: crnn_test
Output file: recognition_test.txt

Recognition data preparation completed!
Training crops: 25693
Test crops: 7206

You now have:
1. Detection data: train_detection.txt, test_detection.txt
2. Recognition data: recognition_train.txt, recognition_test.txt
3. Cropped images: crnn_train/, crnn_test/


In [16]:
!wget -P pretrained_models/detection/ https://paddleocr.bj.bcebos.com/pretrained/MobileNetV3_large_x0_5_pretrained.pdparams

--2026-01-12 16:03:00--  https://paddleocr.bj.bcebos.com/pretrained/MobileNetV3_large_x0_5_pretrained.pdparams
Resolving paddleocr.bj.bcebos.com (paddleocr.bj.bcebos.com)... 103.235.47.176, 2402:2b40:7000:628:0:ff:b0e8:88da
Connecting to paddleocr.bj.bcebos.com (paddleocr.bj.bcebos.com)|103.235.47.176|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3189601 (3.0M) [application/octet-stream]
Saving to: ‘pretrained_models/detection/MobileNetV3_large_x0_5_pretrained.pdparams.1’

MobileNetV3_large_x 100%[===================>]   3.04M   688KB/s    in 17s     

2026-01-12 16:03:19 (186 KB/s) - ‘pretrained_models/detection/MobileNetV3_large_x0_5_pretrained.pdparams.1’ saved [3189601/3189601]



In [17]:
!wget -P pretrained_models/recognition/ https://paddleocr.bj.bcebos.com/PP-OCRv3/english/en_PP-OCRv3_rec_train.tar

--2026-01-12 16:03:19--  https://paddleocr.bj.bcebos.com/PP-OCRv3/english/en_PP-OCRv3_rec_train.tar
Resolving paddleocr.bj.bcebos.com (paddleocr.bj.bcebos.com)... 103.235.47.176, 2402:2b40:7000:628:0:ff:b0e8:88da
Connecting to paddleocr.bj.bcebos.com (paddleocr.bj.bcebos.com)|103.235.47.176|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 204093440 (195M) [application/x-tar]
Saving to: ‘pretrained_models/recognition/en_PP-OCRv3_rec_train.tar’

en_PP-OCRv3_rec_tra 100%[===================>] 194.64M  11.3MB/s    in 33s     

2026-01-12 16:03:54 (5.88 MB/s) - ‘pretrained_models/recognition/en_PP-OCRv3_rec_train.tar’ saved [204093440/204093440]



In [18]:
!tar -xf pretrained_models/recognition/en_PP-OCRv3_rec_train.tar -C pretrained_models/recognition && rm -rf pretrained_models/recognition/en_PP-OCRv3_rec_train.tar

In [19]:
# !python3 PaddleOCR/tools/train.py -c configs/detection_config.yml

In [20]:

# !rm -r PaddleOCR
# !rm -r crnn_test
# !rm -r crnn_train



In [21]:
# import shutil
# from IPython.display import FileLink, FileLinks
# !rm -r my_folder_backup.zip
# !zip -r my_folder_backup.zip /kaggle/working/Fine-tune/output/vn_recognition_2



In [22]:
!python3 PaddleOCR/tools/train.py -c configs/recognition_config.yml \
  -o Global.checkpoints=./output/vn_recognition_2/latest

/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:718: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Skipping import of the encryption module.
[2026/01/12 16:04:07] ppocr INFO: Architecture : 
[2026/01/12 16:04:07] ppocr INFO:     Backbone : 
[2026/01/12 16:04:07] ppocr INFO:         last_conv_stride : [1, 2]
[2026/01/12 16:04:07] ppocr INFO:         last_pool_kernel_size : [2, 2]
[2026/01/12 16:04:07] ppocr INFO:         last_pool_type : avg
[2026/01/12 16:04:07] ppocr INFO:         name : MobileNetV1Enhance
[2026/01/12 16:04:07] ppocr INFO:         scale : 0.5
[2026/01/12 16:04:07] ppocr INFO:     Head : 
[2026/01/12 16:04:07] ppocr INFO:         head_list : 
[2026/01/12 16:04:07] ppocr INFO:             CTCHead : 
[2026/01/12 16:04:07] ppocr INFO:                 H

In [23]:
!python3 PaddleOCR/tools/export_model.py -c configs/recognition_config.yml -o Global.pretrained_model=output/vn_recognition/latest.pdparams  Global.save_inference_dir=models/recognition

/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:718: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Skipping import of the encryption module.
W0112 22:56:58.775483  2058 gpu_resources.cc:114] Please NOTE: device: 0, GPU Compute Capability: 7.5, Driver API Version: 12.8, Runtime API Version: 12.9
[2026/01/12 22:56:59] ppocr WARNING: The shape of model params head.ctc_head.fc.weight [64, 30546] not matched with loaded params head.ctc_head.fc.weight [64, 97] !
[2026/01/12 22:56:59] ppocr WARNING: The shape of model params head.ctc_head.fc.bias [30546] not matched with loaded params head.ctc_head.fc.bias [97] !
[2026/01/12 22:56:59] ppocr WARNING: The shape of model params head.sar_head.decoder.embedding.weight [30548, 512] not matched with loaded params head.sar_head.de

In [24]:
!python3 PaddleOCR/tools/export_model.py \
  -c configs/detection_config.yml \
  -o Global.pretrained_model=output/vn_detection/best_accuracy \
     Global.save_inference_dir=./models/detection \
     Global.checkpoints=null

/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:718: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Skipping import of the encryption module.
W0112 22:57:06.223011  2069 gpu_resources.cc:114] Please NOTE: device: 0, GPU Compute Capability: 7.5, Driver API Version: 12.8, Runtime API Version: 12.9
[2026/01/12 22:57:06] ppocr INFO: load pretrain successful from output/vn_detection/best_accuracy
[2026/01/12 22:57:06] ppocr INFO: Export inference config file to ./models/detection/inference.yml
Skipping import of the encryption module
W0112 22:57:06.971444  2069 eager_utils.cc:3441] Paddle static graph(PIR) not support input out tensor for now!!!!!
[2026/01/12 22:57:08] ppocr INFO: inference model is saved to ./models/detection/inference


detection_model='/kaggle/working/models/detection'
recognition_model='/kaggle/working/models/recognition'

In [25]:
!python3 /kaggle/working/Fine-tune/PaddleOCR/tools/eval.py \
        -c /kaggle/working/Fine-tune/configs/recognition_config.yml  \
        -o Global.checkpoints=/kaggle/working/Fine-tune/output/vn_recognition_2/latest \
                         Global.character_type=ch  \
                         Global.character_dict_path=/kaggle/working/Fine-tune/vn_dictionary.txt

/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:718: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Skipping import of the encryption module.
[2026/01/12 22:57:15] ppocr INFO: Architecture : 
[2026/01/12 22:57:15] ppocr INFO:     Backbone : 
[2026/01/12 22:57:15] ppocr INFO:         last_conv_stride : [1, 2]
[2026/01/12 22:57:15] ppocr INFO:         last_pool_kernel_size : [2, 2]
[2026/01/12 22:57:15] ppocr INFO:         last_pool_type : avg
[2026/01/12 22:57:15] ppocr INFO:         name : MobileNetV1Enhance
[2026/01/12 22:57:15] ppocr INFO:         scale : 0.5
[2026/01/12 22:57:15] ppocr INFO:     Head : 
[2026/01/12 22:57:15] ppocr INFO:         head_list : 
[2026/01/12 22:57:15] ppocr INFO:             CTCHead : 
[2026/01/12 22:57:15] ppocr INFO:                 H

In [26]:
recognition_model = '/kaggle/input/models/models/recognition'
detection_model = '/kaggle/input/models/models/detection'

In [27]:
!python3 /kaggle/working/PaddleOCR/tools/infer/predict_system.py \
    --det_algorithm="DB" \
    --det_model_dir="/kaggle/input/models/models/detection" \
    --rec_model_dir="/kaggle/input/models/models/recognition" \
    --rec_char_dict_path="/kaggle/input/diction/vn_dictionary.txt" \
    --rec_image_shape="3,32,320" \
    --image_dir="/kaggle/input/testdata/11.jpeg" \
    --use_gpu=True 

python3: can't open file '/kaggle/working/PaddleOCR/tools/infer/predict_system.py': [Errno 2] No such file or directory


In [28]:
!python3 /kaggle/working/Fine-tune/PaddleOCR/tools/infer/predict_det.py \
    --det_algorithm="DB"\
    --det_model_dir="/kaggle/input/models/models/detection" \
    --image_dir="/kaggle/input/testdata/11.jpeg" \
    --use_gpu=True

/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:718: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Traceback (most recent call last):
  File "/kaggle/working/Fine-tune/PaddleOCR/tools/infer/predict_det.py", line 413, in <module>
    image_file_list = get_image_file_list(args.image_dir)
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/kaggle/working/Fine-tune/PaddleOCR/ppocr/utils/utility.py", line 80, in get_image_file_list
    raise Exception("not found any img file in {}".format(img_file))
Exception: not found any img file in /kaggle/input/testdata/11.jpeg


In [30]:
!python3 /kaggle/working/PaddleOCR/tools/infer/predict_system.py \
    --det_algorithm="DB" \
    --rec_algorithm="SVTR_LCNet" \
    --det_model_dir="/kaggle/input/models/models/detection" \
    --rec_model_dir="/kaggle/input/models/models/recognition" \
    --rec_char_dict_path="/kaggle/input/diction/vn_dictionary.txt" \
    --image_dir="/kaggle/input/dataset/dataset/train_images/im0004.jpg" \
    --use_gpu=True

python3: can't open file '/kaggle/working/PaddleOCR/tools/infer/predict_system.py': [Errno 2] No such file or directory
